In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. Create preQC_df, do QC offline
2. Load QC; save maybes, mergers, & clean neur figs in the data/units dir
3. Create neur_df with columns for spikes, region, coordinates
4. Merge clusters as needed
5. Inspect & save

### variables

In [2]:
patient = 18
save = False

units_dir = f'../../data/units'
units_all_dir = f'{units_dir}/all/2025{patient}'
units_waves_dir = f'{units_dir}/waves/2025{patient}'
units_clean_dir = f'{units_dir}/clean/2025{patient}'
units_maybes_dir = f'{units_dir}/maybes/2025{patient}'
units_mergers_dir = f'{units_dir}/mergers/2025{patient}'
for dir in [units_all_dir, units_waves_dir, units_clean_dir, units_maybes_dir, units_mergers_dir]:
    os.makedirs(dir, exist_ok=True)

### 1. set up preQC

In [3]:
# save = True

# parse osort figs
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs/5/*'):

    # grab individual CL figs
    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file):
        
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {dest}/')

    # grab WAVES figs to check for mergers
    if 'WAVES' in os.path.basename(file):
        
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {dest}/')

save = False # reset to be safe

### create preQC_df
note: clustID called unitID going forward

In [4]:
# init preQC_df 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# parse units_all_dir
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:]))
    unitIDs.append(int(os.path.basename(file).split('_')[2]))

# create df
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

# save df
preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if not os.path.exists(preQC_df_path):
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)

save = False # reset to be safe
preQC_df

,chanID,unitID
0,193,421
1,193,467
2,194,179
3,196,308
4,197,203
...,...,...
65,221,3
66,222,4
67,223,2
68,223,3


### 2. load QC; save maybes, clean, mergers, and possible_neurs (clean+merger) neur figs in the data/units dir

In [5]:
save=True

QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')
maybes_df = QC_df[~QC_df['notes'].isna()].copy().reset_index(drop=True)
clean_df = QC_df[QC_df['keep'] == 1].copy().reset_index(drop=True)
mergers_df = QC_df[~QC_df['merge_cluster'].isna()].copy().reset_index(drop=True)

# clean + mergers
possible_neurs_df = pd.concat([clean_df, mergers_df]).drop_duplicates().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    chanID = int(os.path.basename(unit_file).split('_')[0][1:])
    unitID = int(os.path.basename(unit_file).split('_')[2])
    
    if ((maybes_df['chanID'] == chanID) & (maybes_df['unitID'] == unitID)).any():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

    if ((mergers_df['chanID'] == chanID) & (mergers_df['unitID'] == unitID)).any():
        dest = f'{units_mergers_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

    if ((clean_df['chanID'] == chanID) & (clean_df['unitID'] == unitID)).any():
        dest = f'{units_clean_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

assert len(clean_df) == len(glob.glob(f'{units_clean_dir}/*.png')), f'Length mismatch: clean_df has {len(clean_df)} rows, but {len(glob.glob(f"{units_clean_dir}/*.png"))} files in {units_clean_dir}'
assert len(maybes_df) == len(glob.glob(f'{units_maybes_dir}/*.png')), f'Length mismatch: maybes_df has {len(maybes_df)} rows, but {len(glob.glob(f"{units_maybes_dir}/*.png"))} files in {units_maybes_dir}'
assert len(mergers_df) == len(glob.glob(f'{units_mergers_dir}/*.png')), f'Length mismatch: mergers_df has {len(mergers_df)} rows, but {len(glob.glob(f"{units_mergers_dir}/*.png"))} files in {units_mergers_dir}'

save = False # reset to be safe
possible_neurs_df

,chanID,unitID,keep,merge_cluster,notes
0,193,467,1.0,NaN,NaN
1,202,871,1.0,NaN,NaN
2,204,701,1.0,1.0,NaN
3,204,708,1.0,1.0,NaN
4,205,590,1.0,NaN,NaN
5,206,612,1.0,NaN,NaN
6,207,1988,1.0,2.0,NaN
7,210,4424,1.0,NaN,NaN
8,210,4457,1.0,NaN,NaN
9,210,4546,1.0,NaN,NaN


### 3. create neur_df for neurs in possible_neurs_df and taking in spike times from osort mat files

### helpers

In [6]:
def getunitID2spikes(unitIDs, spikes, possible_neurs_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID not in possible_neurs_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [7]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, possible_neurs_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
neur_df = pd.merge(
    neur_df,
    possible_neurs_df[['chanID', 'unitID', 'keep', 'merge_cluster']],
    on=['chanID', 'unitID'],
    how='left'
)
neur_df = neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
assert len(neur_df) == len(possible_neurs_df), f'Length mismatch: neur_df has {len(neur_df)} rows, possible_neurs_df has {len(possible_neurs_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster
0,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,1.0,NaN
1,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,1.0,NaN
2,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810,1.0,1.0
3,204,708,"[5.711866666666667, 12.2751, 13.62493333333333...",1026,0.622157,1.0,1.0
4,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,1.0,NaN
5,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,1.0,NaN
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,1.0,2.0
7,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241,NaN,2.0
8,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,1.0,NaN
9,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,1.0,NaN


### add region column by mapping channel -> region (label)

In [8]:
if patient != 22:
    chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

    # variable across patients
    if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
    elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

    labelMap = chanMap['LabelMap'].flatten() # contains region labels
    labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

    nan_mask = ~np.isnan(channelMap)
    channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

    neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))

else: neur_df['region'] = np.arange(len( neur_df)) # for pt22, just assign region = chanID since no chanMap available

neur_df


,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster,region
0,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,1.0,NaN,mROFC1
1,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,1.0,NaN,mRACC2
2,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810,1.0,1.0,mRACC4
3,204,708,"[5.711866666666667, 12.2751, 13.62493333333333...",1026,0.622157,1.0,1.0,mRACC4
4,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,1.0,NaN,mRACC5
5,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,1.0,NaN,mRACC6
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,1.0,2.0,mRACC7
7,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241,NaN,2.0,mRACC7
8,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,1.0,NaN,mRAHIP2
9,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,1.0,NaN,mRAHIP2


### add coordinates columns by mapping region->coords

In [9]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

In [10]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
# ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?
# atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
# atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
if patient == 22:
    neur_df = neur_df.copy()
    neur_df[['x', 'y', 'z']] = 0
else:
    neur_df = (neur_df
                    .merge(region2id_df, on='region', how='left')
                    .merge(id2xyz_df, on='ID', how='left'))
                  #   .merge(xyz2atlasRegions, on='ID', how='left')
    neur_df = neur_df.drop(columns=['ID'])
    
# sort by chanID
neur_df = neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
neur_df

,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster,region,x,y,z
0,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,1.0,NaN,mROFC1,5.400004,42.999992,-10.400004
1,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,1.0,NaN,mRACC2,6.600004,29.799992,23.199994
2,204,701,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",1727,1.043810,1.0,1.0,mRACC4,7.800004,27.399993,23.199994
3,204,708,"[5.711866666666667, 12.2751, 13.62493333333333...",1026,0.622157,1.0,1.0,mRACC4,7.800004,27.399993,23.199994
4,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,1.0,NaN,mRACC5,7.800004,29.799992,23.199994
5,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,1.0,NaN,mRACC6,9.000004,28.599993,23.199994
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,1.0,2.0,mRACC7,9.000004,27.399993,23.199994
7,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241,NaN,2.0,mRACC7,9.000004,27.399993,23.199994
8,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,1.0,NaN,mRAHIP2,16.600004,-3.000006,-16.400003
9,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,1.0,NaN,mRAHIP2,16.600004,-3.000006,-16.400003


### 4. Merge clusters as needed

In [11]:
merged_rows = []
merge_mask = neur_df['merge_cluster'].notna()

# extract merger rows as df
for _, merge_group in neur_df[merge_mask].groupby('merge_cluster'):
    merged_spikes = np.sort(np.concatenate(merge_group['spikes'].values))

    # update row with merged info
    first_row = merge_group.iloc[0]
    merged_rows.append({
        **{col: first_row[col] for col in neur_df.columns},
        'unitID':     '/'.join(merge_group['unitID'].astype(str).tolist()),
        'spikes':     merged_spikes,
        'num_spikes': len(merged_spikes),
        'FR':         len(merged_spikes) / (merged_spikes[-1] - merged_spikes[0]),
        'keep':       1,
        # chanID, region, merge_cluster, x, y, z already from iloc[0]
    })

# concat original df without merged rows + new merged rows
neur_df = pd.concat(
    [neur_df[~merge_mask], pd.DataFrame(merged_rows)],
    ignore_index=True
)
neur_df = neur_df.sort_values(by=['chanID']).reset_index(drop=True)
neur_df


,chanID,unitID,spikes,num_spikes,FR,keep,merge_cluster,region,x,y,z
0,193,467,"[3.260966666666667, 5.094733333333334, 11.0821...",1250,0.757284,1.0,NaN,mROFC1,5.400004,42.999992,-10.400004
1,202,871,"[3.734466666666667, 4.718, 8.183800000000002, ...",1114,0.681801,1.0,NaN,mRACC2,6.600004,29.799992,23.199994
2,204,701/708,"[0.5282, 1.0729333333333335, 1.3612, 2.3284666...",2753,1.663931,1.0,1.0,mRACC4,7.800004,27.399993,23.199994
3,205,590,"[3.493566666666667, 4.5576, 7.251833333333334,...",912,0.552273,1.0,NaN,mRACC5,7.800004,29.799992,23.199994
4,206,612,"[0.017533333333333335, 0.6868666666666667, 3.7...",951,0.576399,1.0,NaN,mRACC6,9.000004,28.599993,23.199994
5,207,1988/2001,"[0.25660000000000005, 0.5228666666666667, 0.72...",9182,5.518271,1.0,2.0,mRACC7,9.000004,27.399993,23.199994
6,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,1.0,NaN,mRAHIP2,16.600004,-3.000006,-16.400003
7,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,1.0,NaN,mRAHIP2,16.600004,-3.000006,-16.400003
8,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,1.0,NaN,mRAHIP2,16.600004,-3.000006,-16.400003
9,211,2553/2862,"[2.716666666666667, 2.723366666666667, 2.73293...",2152,1.296604,1.0,3.0,mRAHIP3,16.600004,-0.600006,-16.400003


### 5. inspect, typecast, & save

In [12]:
neur_df['spikes'] = neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays
print('example neuron')
print(f'last 5 spikes (s): {neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {neur_df["spikes"].iloc[0][-5:]/60}')

neur_df['unitID'] = neur_df['unitID'].astype(str)
 
# add patient as first column
if 'patient' not in neur_df: neur_df.insert(0, 'patient', patient)

# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if os.path.exists(parquet_path):     print(f'File already exists, skipping: {parquet_path}')
else:     neur_df.to_parquet(parquet_path, index=False)
neur_df.to_parquet(parquet_path, index=False)

example neuron
last 5 spikes (s): [1639.3013 1640.7237 1642.3562 1652.0312 1653.8973]
last 5 spikes (min): [27.32168833 27.345395   27.37260333 27.53385333 27.564955  ]
File already exists, skipping: ../../results/202518/records/processed_data/df_neurs.parquet
